In [1]:
# regression problem (p,H)->RO
# includes neural network definition and training
# no phase separation - for better accuracy use phase separation

# CAUTION: additional transformation of the target vector: y-min(rho)
#          scaling constant stored as an additional parameter 'ro_min' in the pickle file
#          actual density computed as rho=y_pred*yst+ro_min, where y_pred is the model prediciton
#          additional feature: p**(1/3)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import time
from sklearn.model_selection import train_test_split
import sys

import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
#data preparation
#data import
col=['PT', 'H','CO2', 'CO', 'H2', 'N2', 'Ar', 'RS', '2phase','ROG','ROHL']
data = pd.read_csv("../michal/ML_pytroch/Data/GERG2008_1m.csv",
 encoding="utf-8", nrows=1_000_000, usecols=col)

#phase separation
PH=(2*data['2phase']**0).copy()
PH[data.RS==1]=1
PH[data.RS==0]=0
data['PH']=PH
#computation of the total density
data['RO'] = 1 / (data.RS / data.ROG + (1 - data.RS) / data.ROHL)

In [4]:
# regression problem

#feature matrix
X=data.drop(columns=['PH','RS','2phase','ROG', 'ROHL', 'RO'])
X['P2']=X['PT']**(1/3)

#target vector
ro_min=data['RO'].min()
y=data.RO-ro_min 

yst=y.std()
yy=y/yst



X_trr, X_tss, y_trr, y_tss = train_test_split(X, yy, test_size=0.3, random_state=42)

#scaling
xtm=X_trr.mean()
xts=X_trr.std()


X_tr=torch.tensor(np.array((X_trr-xtm)/xts), dtype=torch.float32)
y_tr=torch.tensor(np.array(y_trr), dtype=torch.float32).reshape(-1, 1)
X_ts=torch.tensor(np.array((X_tss-xtm)/xts), dtype=torch.float32)
y_ts=torch.tensor(np.array(y_tss), dtype=torch.float32).reshape(-1, 1)

model = nn.Sequential(
    nn.Linear(len(X_tr[0,:]), 20),
    nn.ReLU(),
    nn.Linear(20, 20),
    nn.ReLU(),
    nn.Linear(20, 1))

loss_fn=nn.MSELoss()
#loss_fn=nn.L1Loss()

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0)

n_epochs = 300
batch_size = 200

lls=[] 
for epoch in range(n_epochs):
    if epoch>150:
        optimizer = optim.Adam(model.parameters(), lr=0.0005)
    for i in range(0, len(X_tr), batch_size):
        Xbatch = X_tr[i:i+batch_size]
        ybatch = y_tr[i:i+batch_size]
        y_pred = model(Xbatch)
        loss = loss_fn(y_pred, ybatch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Finished epoch {epoch}, latest loss {loss}')
    lls.append(loss.item())
    
with torch.no_grad():
    y_pred_train = (model(X_tr))
    y_pred_test = (model(X_ts))
    
print("MSE train: {0:}, test: {1:}".\
      format(mean_squared_error(y_tr, y_pred_train), 
             mean_squared_error(y_ts, y_pred_test)))
print("R^2 train: {0:}, test: {1:}".\
      format(r2_score(y_tr, y_pred_train),
             r2_score(y_ts, y_pred_test)))

b_train=np.array(abs((y_tr-y_pred_train)/(y_tr+ro_min)))
b_test=np.array(abs((y_ts-y_pred_test)/(y_ts+ro_min)))
print('max error train: ',b_train.max(), 'mean error train: ',b_train.mean())
print('max error test: ',b_test.max(), 'mean error test: ',b_test.mean())

Finished epoch 0, latest loss nan
Finished epoch 1, latest loss nan
Finished epoch 2, latest loss nan
Finished epoch 3, latest loss nan
Finished epoch 4, latest loss nan
Finished epoch 5, latest loss nan
Finished epoch 6, latest loss nan
Finished epoch 7, latest loss nan
Finished epoch 8, latest loss nan
Finished epoch 9, latest loss nan
Finished epoch 10, latest loss nan
Finished epoch 11, latest loss nan
Finished epoch 12, latest loss nan
Finished epoch 13, latest loss nan
Finished epoch 14, latest loss nan
Finished epoch 15, latest loss nan
Finished epoch 16, latest loss nan
Finished epoch 17, latest loss nan
Finished epoch 18, latest loss nan
Finished epoch 19, latest loss nan
Finished epoch 20, latest loss nan
Finished epoch 21, latest loss nan
Finished epoch 22, latest loss nan
Finished epoch 23, latest loss nan
Finished epoch 24, latest loss nan
Finished epoch 25, latest loss nan
Finished epoch 26, latest loss nan
Finished epoch 27, latest loss nan
Finished epoch 28, latest loss

ValueError: Input contains NaN.

In [ ]:
# results visualization

# learning curve
plt.plot(np.log(lls))
plt.xlabel('epoch')
plt.ylabel('log(loss)')
plt.show()

#relative error of test data
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(X_tss.PT,X_tss.H, b_test,c=b_test)
ax.set_title('relative error of test data')
ax.set_xlabel('pressure [MPa]')
ax.set_ylabel('enthalpy [J/(kg*K)]')
#ax.set_zlim([0,1])
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(X_tss.PT, X_tss.H, y_ts+ro_min,c=PH.loc[X_tss.index])
ax.set_title('test data')
ax.set_xlabel('pressure [MPa]')
ax.set_ylabel('enthalpy [J/(kg*K)]')
plt.show()

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(X_tss.PT, X_tss.H, y_pred_test+ro_min,c=PH.loc[X_tss.index])
ax.set_title('test data - prediction')
ax.set_xlabel('pressure [MPa]')
ax.set_ylabel('enthalpy [J/(kg*K)]')
plt.show()

In [ ]:
import pickle
pickle.dump([xtm,xts,yst,model,ro_min],open('model_regression_PH_RO','wb'))